In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import svm
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import os

## Data Preparation

In [4]:
csv_files = ['ampc/w1.csv', 'ampc/w2.csv', 'ampc/w3.csv', 'ampc/w4.csv']

dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)
    
combined_data = pd.concat(dfs, ignore_index=True)
combined_data.to_csv('combined_data.csv', index=False)

In [5]:
shuffled_data = combined_data.sample(frac=1, random_state=42)
shuffled_data.to_csv('all_data.csv', index=False)


In [6]:
shuffled_data.head()

,acc_mean_x_right,acc_mean_y_right,acc_mean_z_right,acc_mean_xyz_right,acc_mean_xy_right,acc_mean_yz_right,acc_mean_zx_right,acc_mean_pitch_right,acc_mean_roll_right,acc_std_x_right,...,gyro_max_yz_left,gyro_max_zx_left,gyro_peak_x_left,gyro_peak_y_left,gyro_peak_z_left,gyro_peak_xyz_left,gyro_peak_xy_left,gyro_peak_yz_left,gyro_peak_zx_left,class
1275,-0.47189,0.30558,0.831100,1.0034,0.56224,0.88554,0.95576,-40.481,53.233,0.006978,...,1.589,1.0254,9,8,8,8,7,8,9,0
5079,0.84220,0.35072,0.308980,1.2641,1.05270,0.74039,1.13660,71.101,32.668,0.446530,...,271.980,154.8600,7,6,4,6,6,6,5,2
1904,-0.98024,0.52331,0.061482,1.3013,1.17830,0.74973,1.13200,-56.072,35.152,0.369990,...,50.073,76.1390,4,5,4,4,4,5,4,2
1039,-0.20462,0.33777,0.900730,1.1621,0.62003,1.05140,1.03660,-14.699,54.450,0.478460,...,171.610,208.9000,3,5,3,2,2,4,3,2
11294,-0.33887,-0.81768,0.443720,1.1962,1.08860,1.01700,0.72398,-12.913,-49.806,0.493840,...,252.480,163.4900,2,4,1,4,5,5,2,2


## Model Training 

In [8]:
all_data = pd.read_csv('all_data.csv')

# Separate features and class
X = all_data.iloc[:, :-1]  # All columns except the last one
y = all_data.iloc[:, -1]   # Only the last column

# a. Train-test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

clf = svm.SVC()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
train_test_accuracy = accuracy_score(y_test, y_pred)

print(f"Train-test split accuracy: {train_test_accuracy:.4f}")

# b. 10-fold cross-validation
scores = cross_val_score(clf, X, y, cv=10)
cv_accuracy = scores.mean()

print(f"10-fold cross-validation accuracy: {cv_accuracy:.4f}")
print(f"Cross-validation scores: {scores}")

Train-test split accuracy: 0.8882
10-fold cross-validation accuracy: 0.8918
Cross-validation scores: [0.90283749 0.89423904 0.89251935 0.88392089 0.89595873 0.89595873
 0.88048151 0.89337919 0.90025795 0.87865749]


## Hyperparameter tuning

In [10]:
clf = svm.SVC()

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

grid = GridSearchCV(svm.SVC(), param_grid, refit=True, verbose=2, cv=5)
grid.fit(X, y) 

 


Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  19.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  19.1s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  19.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  19.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=  19.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  19.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  21.4s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  19.2s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  21.7s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=  20.3s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=  20.7s
[CV] END ......................C=0.1, gamma=0.01

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001],
                         'kernel': ['rbf']},
             verbose=2)

In [11]:
print("Best parameters found: ", grid.best_params_)
print("Best accuracy: ", grid.best_score_)

Best parameters found:  {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
Best accuracy:  0.8356694496065977


In [12]:
best_params = grid.best_params_
best_svm = svm.SVC(**best_params)

# a. Train-test split with hyperparameter tuning
best_svm.fit(X_train, y_train)
y_pred_best = best_svm.predict(X_test)
best_train_test_accuracy = accuracy_score(y_test, y_pred_best)

# b. 10-fold cross-validation with hyperparameter tuning
best_scores = cross_val_score(best_svm, X, y, cv=10)
best_cv_accuracy = best_scores.mean()

## Feature Selection

In [14]:
selector = SelectKBest(f_classif, k=100)
X_new = selector.fit_transform(X, y)

selected_indices = selector.get_support(indices=True)
selected_feature_names = X.columns[selected_indices]

# a. Train-test split (70/30) with hyperparameter tuning
X_train_selected, X_test_selected, y_train, y_test = train_test_split(X_new, y, test_size=0.3, random_state=1)

# Train SVM with hyperparameter tuning on selected features
best_svm_selected = svm.SVC(**best_params)
best_svm_selected.fit(X_train_selected, y_train)
y_pred_selected = best_svm_selected.predict(X_test_selected)
selected_train_test_accuracy = accuracy_score(y_test, y_pred_selected)

# 10-fold cross-validation with feature selection
selected_scores = cross_val_score(best_svm_selected, X_new, y, cv=10)
selected_cv_accuracy = selected_scores.mean()

## Dimensionality Reduction

In [16]:
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X)

# a. Train-test split (70/30) with hyperparameter tuning
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.3, random_state=1)

# Train SVM with hyperparameter tuning on PCA-transformed data
best_svm_pca = svm.SVC(**best_params)
best_svm_pca.fit(X_train_pca, y_train)
y_pred_pca = best_svm_pca.predict(X_test_pca)
pca_train_test_accuracy = accuracy_score(y_test, y_pred_pca)

# 10-fold cross-validation with PCA
pca_scores = cross_val_score(best_svm_pca, X_pca, y, cv=10)
pca_cv_accuracy = pca_scores.mean()



## Summary Table

In [18]:
svm_summary = pd.DataFrame({
    'Model': ['Original features', 
              'With hyper-parameter tuning',
              'With feature selection and hyper-parameter tuning',
              'With PCA and hyper-parameter tuning'],
    'Train-test split': [f"{train_test_accuracy:.2%}", 
                         f"{best_train_test_accuracy:.2%}",
                         f"{selected_train_test_accuracy:.2%}",
                         f"{pca_train_test_accuracy:.2%}"],
    'Cross-validation': [f"{cv_accuracy:.2%}", 
                          f"{best_cv_accuracy:.2%}",
                          f"{selected_cv_accuracy:.2%}",
                          f"{pca_cv_accuracy:.2%}"]
})

print("SVM Model Summary Table:")
svm_summary

SVM Model Summary Table:


,Model,Train-test split,Cross-validation
0,Original features,88.82%,89.18%
1,With hyper-parameter tuning,83.69%,83.57%
2,With feature selection and hyper-parameter tuning,83.95%,83.75%
3,With PCA and hyper-parameter tuning,83.78%,83.58%


## Other Classifiers 

In [20]:
# 1. SGD Classifier
sgd = SGDClassifier(max_iter=1000, random_state=42)

# Train-test split
sgd.fit(X_train, y_train)
y_pred_sgd = sgd.predict(X_test)
sgd_train_test_accuracy = accuracy_score(y_test, y_pred_sgd)

# Cross-validation
sgd_scores = cross_val_score(sgd, X, y, cv=10)
sgd_cv_accuracy = sgd_scores.mean()

# 2. Random Forest Classifier
rf = RandomForestClassifier(random_state=42)

# Train-test split
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_train_test_accuracy = accuracy_score(y_test, y_pred_rf)

# Cross-validation
rf_scores = cross_val_score(rf, X, y, cv=10)
rf_cv_accuracy = rf_scores.mean()

# 3. MLP Classifier
mlp = MLPClassifier(max_iter=1000, random_state=42)

# Train-test split
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
mlp_train_test_accuracy = accuracy_score(y_test, y_pred_mlp)

# Cross-validation
mlp_scores = cross_val_score(mlp, X, y, cv=10)
mlp_cv_accuracy = mlp_scores.mean()


In [21]:
classifier_summary = pd.DataFrame({
    'Model': ['SVM', 'SGD', 'RandomForest', 'MLP'],
    'Train-test split': [f"{best_train_test_accuracy:.2%}", 
                         f"{sgd_train_test_accuracy:.2%}",
                         f"{rf_train_test_accuracy:.2%}",
                         f"{mlp_train_test_accuracy:.2%}"],
    'Cross-validation': [f"{best_cv_accuracy:.2%}", 
                         f"{sgd_cv_accuracy:.2%}",
                         f"{rf_cv_accuracy:.2%}",
                         f"{mlp_cv_accuracy:.2%}"]
})

print("Classifier Summary Table:")
classifier_summary

Classifier Summary Table:


,Model,Train-test split,Cross-validation
0,SVM,83.69%,83.57%
1,SGD,88.25%,87.83%
2,RandomForest,92.35%,92.56%
3,MLP,89.54%,82.73%
